In [ ]:
########=======MONTHLY QUANTILE MAPPING BIAS CORRECTION FOR SRPs========#############
##================Replace CHIRPS with other SRPs, e.g., MSWEP, TAMSAT, etc.============####


#!/usr/bin/env python
# coding: utf-8

import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import glob
import os
from sklearn.metrics import mean_squared_error
import logging
from scipy.ndimage import gaussian_filter

# ============================
# SETUP
# ============================
logging.basicConfig(filename=os.path.join(r"D:\Working_directory", "ls_script.log"),
                    level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")

csv_dir = r"D:\Path_to_CSV_folder"             #station csv files with columns for 'Date', 'Insitu" and 'SRP" rainfall values
coords_path = r"D:\Path_to_corrdinates_file\Coordinates.csv" #Coordinates file with 'Station name', 'Longitude' and 'Latitude' columns 
chirps_folder = r"D:\Path_to_SRP_raster_files"   #Folder containing SRPs raster files
output_folder = r"D:\OUTPUT_LS"
sample_raster_path = r"D:\Path_to_SRP_raster_files\P_CHIRPS.v2.0_mm-month-1_monthly_2012.01.01.tif"  #Sample raster file to use as reference for CRS, resolution, extent, etc.

os.makedirs(output_folder, exist_ok=True)

# ============================
# NSE FUNCTION
# ============================
def calculate_nse(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)
    mean_observed = np.mean(observed)
    numerator = np.sum((observed - predicted) ** 2)
    denominator = np.sum((observed - mean_observed) ** 2)
    return 1 - (numerator / denominator) if denominator != 0 else np.nan

# ============================
# QM FUNCTION
# ============================
def qm_correction(obs, sim):
    valid_mask = ~(np.isnan(obs) | np.isnan(sim))
    if np.sum(valid_mask) < 3:
        return sim, 1.0

    obs_valid = obs[valid_mask]
    sim_valid = sim[valid_mask]

    obs_sorted = np.sort(obs_valid)
    ranks = sim_valid.argsort().argsort()
    quantiles = ranks / (len(sim_valid) - 1)
    corrected_valid = np.quantile(obs_sorted, quantiles)

    corrected = np.full_like(sim, np.nan)
    corrected[valid_mask] = corrected_valid
    corrected = np.maximum(corrected, 0)

    ratio = corrected_valid / sim_valid
    ratio = ratio[np.isfinite(ratio) & (sim_valid > 0)]
    factor = np.median(ratio) if len(ratio) > 0 else 1.0

    return corrected, factor

# ============================
# STEP 1: LOAD DATA
# ============================
csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))
station_data = {}

for file in csv_files:
    df = pd.read_csv(file, parse_dates=["Date"])
    df = df[(df["InSitu"] >= 0) & (df["CHIRPS"] >= 0) | df["InSitu"].isna() | df["CHIRPS"].isna()]
    station = os.path.basename(file).replace(".csv", "").strip("_")
    station_data[station] = df

# ============================
# STEP 2: APPLY QM PER MONTH
# ============================
monthly_cfs = {}
valid_months = {}

for station, df in station_data.items():
    df["Month"] = df["Date"].dt.month
    df["Corrected"] = np.nan

    monthly_cfs[station] = {}
    valid_months[station] = []

    for month in range(1, 13):
        df_month = df[df["Month"] == month]
        valid_mask = ~(df_month["InSitu"].isna() | df_month["CHIRPS"].isna())
        df_valid = df_month[valid_mask]

        if len(df_valid) >= 3:
            corrected_values, factor = qm_correction(
                df_valid["InSitu"].values,
                df_valid["CHIRPS"].values
            )

            df.loc[df_month.index[valid_mask], "Corrected"] = corrected_values[valid_mask]
            monthly_cfs[station][month] = factor
            valid_months[station].append(month)

        else:
            df.loc[df_month.index, "Corrected"] = df_month["CHIRPS"]
            monthly_cfs[station][month] = 1.0

    df.to_csv(os.path.join(output_folder, f"{station}_corrected.csv"), index=False)

# ============================
# SAVE FACTORS
# ============================
cfs_list = []
for station in monthly_cfs:
    for month in range(1, 13):
        cfs_list.append({
            "Station": station,
            "Month": month,
            "Correction_Factor": monthly_cfs[station][month]
        })

pd.DataFrame(cfs_list).to_csv(os.path.join(output_folder, "correction_factors.csv"), index=False)

# ============================
# STEP 3: VALIDATION
# ============================
metrics_list = []

for station, df in station_data.items():
    valid_df = df[df["Month"].isin(valid_months[station])]
    valid_df = valid_df.dropna(subset=["InSitu", "Corrected"])

    if len(valid_df) < 3:
        continue

    obs = valid_df["InSitu"]
    pred = valid_df["Corrected"]

    corr, _ = pearsonr(obs, pred)
    rmse = np.sqrt(mean_squared_error(obs, pred))
    nse = calculate_nse(obs, pred)

    metrics_list.append({
        "Station": station,
        "Correlation": corr,
        "RMSE": rmse,
        "NSE": nse
    })

pd.DataFrame(metrics_list).to_csv(os.path.join(output_folder, "validation_metrics.csv"), index=False)

# ============================
# STEP 4: IDW + SMOOTHING
# ============================
coords_df = pd.read_csv(coords_path)
x = coords_df["Longitude"].values
y = coords_df["Latitude"].values

with rasterio.open(sample_raster_path) as src:
    transform = src.transform
    width = src.width
    height = src.height
    crs = src.crs

cols, rows = np.meshgrid(np.arange(width), np.arange(height))
xs, ys = rasterio.transform.xy(transform, rows, cols)
xs = np.array(xs).flatten()
ys = np.array(ys).flatten()

def idw(x, y, z, xi, yi, power=2):
    dist = np.sqrt((xi[:, None] - x[None, :])**2 + (yi[:, None] - y[None, :])**2)
    dist[dist == 0] = 1e-12
    w = 1 / dist**power
    w /= np.sum(w, axis=1)[:, None]
    return np.sum(w * z[None, :], axis=1)

monthly_grids = {}

for month in range(1, 13):
    z = np.array([monthly_cfs[s][month] for s in station_data.keys()])
    grid = idw(x, y, z, xs, ys).reshape((height, width))
    grid = gaussian_filter(grid, sigma=2)

    monthly_grids[month] = grid

    with rasterio.open(os.path.join(output_folder, f"CorrectionGrid_Month_{month:02d}_qm_idw_smoothed.tif"),
                       "w", driver="GTiff",
                       height=height, width=width,
                       count=1, dtype=grid.dtype,
                       crs=crs, transform=transform) as dst:
        dst.write(grid, 1)

# ============================
# STEP 5: APPLY TO CHIRPS
# ============================
chirps_files = glob.glob(os.path.join(chirps_folder, "*.tif"))

for file_path in chirps_files:
    base_name = os.path.basename(file_path)
    date_str = base_name.split("_")[-1].replace(".tif", "")
    date = pd.to_datetime(date_str, format="%Y.%m.%d")
    month = date.month

    prefix = base_name.replace(f"_{date_str}.tif", "")
    filename = f"{prefix}_corrected_{date_str}.tif"
    output_path = os.path.join(output_folder, filename)

    with rasterio.open(file_path) as src:
        chirps_data = src.read(1)
        meta = src.meta.copy()

    corrected_data = chirps_data * monthly_grids[month]

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(corrected_data, 1)

print("QM Monthly Bias Correction Completed")


# ============================
# STEP 6: EXAMPLE VALIDATION AT A PIXEL USING THE CORRECTED OUTPUT RASTER
# ============================
try:
    # Get the first corrected output file
    corrected_files = sorted(glob.glob(os.path.join(output_folder, "*_corrected_*.tif")))
    if not corrected_files:
        raise FileNotFoundError("No corrected files found in output folder.")

    sample_corrected_path = corrected_files[0]
    with rasterio.open(sample_corrected_path) as src:
        corrected_data = src.read(1)
        nodata = src.nodata

        # Extract date from filename
        base_name = os.path.basename(sample_corrected_path)
        date_str = base_name.split("_")[-1].replace(".tif", "")  # e.g., "2012.01.01"
        date = pd.to_datetime(date_str, format="%Y.%m.%d")
        month = date.month

        # Also read the original corresponding file for comparison
        original_filename = base_name.replace("_corrected_", "_")  # Reconstruct original name
        original_path = os.path.join(chirps_folder, original_filename)

        if not os.path.exists(original_path):
            print(f"Warning: Original file not found: {original_path}")
            logging.warning(f"Original file not found: {original_path}")
            original_value = "N/A"
        else:
            with rasterio.open(original_path) as src_orig:
                original_data = src_orig.read(1)
                original_value = original_data[0, 0] if nodata is None or original_data[0, 0] != nodata else "NoData"

        corrected_value = corrected_data[0, 0] if nodata is None or corrected_data[0, 0] != nodata else "NoData"

        print(f"Validation using: {base_name}")
        print(f"Date: {date_str}, Month: {month}")
        print(f"Original (top-left pixel): {original_value}")
        print(f"Corrected (top-left pixel): {corrected_value}")
        logging.info(f"Pixel validation - File: {base_name}, Original: {original_value}, Corrected: {corrected_value}")

except Exception as e:
    logging.error(f"Error in pixel validation: {e}")
    print(f"Error in pixel validation: {e}")